# PneumoniaMNIST: Explainable Hybrid Deep Learning and Ensemble Pipeline

This Colab notebook implements a reproducible pipeline for **PneumoniaMNIST** using the compressed `.npz` file directly.

The workflow includes:
1. Google Drive mounting and output-folder creation.
2. Dataset loading from `pneumoniamnist.npz`.
3. Preprocessing and class-distribution analysis.
4. Compact CNN training.
5. CNN embedding extraction.
6. Classical ML models on embeddings: Random Forest, ExtraTrees, XGBoost, LightGBM.
7. Soft-voting ensemble.
8. Evaluation using accuracy, balanced accuracy, precision, recall, F1-score, ROC-AUC, log loss, and ECE.
9. Explainability using Grad-CAM and SHAP on CNN embeddings.
10. Saving all figures, tables, models, and an `outputs_summary.txt` file to Google Drive.

In [ ]:
# ============================================================
# 1. Mount Google Drive and prepare reproducible output folders
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, time, warnings
warnings.filterwarnings("ignore")

PROJECT_NAME = "PneumoniaMNIST_XAI_Hybrid"
BASE_DIR = Path("/content/drive/MyDrive/Outputs") / PROJECT_NAME
FIG_DIR = BASE_DIR / "figures"
TABLE_DIR = BASE_DIR / "tables"
MODEL_DIR = BASE_DIR / "models"
OTHER_DIR = BASE_DIR / "others"

for d in [FIG_DIR, TABLE_DIR, MODEL_DIR, OTHER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

OUTPUT_SUMMARY = BASE_DIR / "outputs_summary.txt"
OUTPUT_SUMMARY.write_text("PneumoniaMNIST XAI Hybrid Pipeline\n", encoding="utf-8")

def log(msg):
    print(msg)
    with open(OUTPUT_SUMMARY, "a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")

log(f"[INFO] Output directory: {BASE_DIR}")

In [ ]:
# ============================================================
# 2. Install required packages
# ============================================================
!pip install -q medmnist xgboost lightgbm shap lime scikit-learn joblib

In [ ]:
# ============================================================
# 3. Imports and reproducibility setup
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score, log_loss,
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    RocCurveDisplay
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

log(f"[INFO] TensorFlow version: {tf.__version__}")

## Dataset path

The notebook first searches common Drive locations. If your file is elsewhere, update `DATA_PATH` manually.

In [ ]:
# ============================================================
# 4. Locate and load PneumoniaMNIST .npz directly
# ============================================================
candidate_paths = [
    "/content/drive/MyDrive/Datasets/pneumoniamnist.npz",
    "/content/drive/MyDrive/Datasets/Pneumonia/pneumoniamnist.npz",
    "/content/drive/MyDrive/Datasets/Medical/pneumoniamnist.npz",
    "/content/drive/MyDrive/Datasets/MedMNIST/pneumoniamnist.npz",
    "/content/drive/MyDrive/Datasets/Heart/pneumoniamnist.npz",
]

DATA_PATH = None
for p in candidate_paths:
    if Path(p).exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    # Fallback: search under MyDrive/Datasets
    matches = list(Path("/content/drive/MyDrive/Datasets").rglob("pneumoniamnist.npz"))
    if matches:
        DATA_PATH = str(matches[0])

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find pneumoniamnist.npz. Please place it under /content/drive/MyDrive/Datasets/ "
        "or update DATA_PATH manually."
    )

log(f"[PATH CHECK] Dataset found: {DATA_PATH}")

data = np.load(DATA_PATH)
log(f"[INFO] Keys in .npz file: {data.files}")

X_train_raw = data["train_images"]
y_train_raw = data["train_labels"]
X_val_raw   = data["val_images"]
y_val_raw   = data["val_labels"]
X_test_raw  = data["test_images"]
y_test_raw  = data["test_labels"]

log(f"[INFO] Train images: {X_train_raw.shape}, labels: {y_train_raw.shape}")
log(f"[INFO] Val images:   {X_val_raw.shape}, labels: {y_val_raw.shape}")
log(f"[INFO] Test images:  {X_test_raw.shape}, labels: {y_test_raw.shape}")

In [ ]:
# ============================================================
# 5. Preprocessing
# ============================================================
def preprocess_images(X):
    X = X.astype("float32") / 255.0
    if X.ndim == 3:
        X = X[..., np.newaxis]
    return X

X_train = preprocess_images(X_train_raw)
X_val   = preprocess_images(X_val_raw)
X_test  = preprocess_images(X_test_raw)

y_train = y_train_raw.reshape(-1).astype(int)
y_val   = y_val_raw.reshape(-1).astype(int)
y_test  = y_test_raw.reshape(-1).astype(int)

CLASS_NAMES = ["Normal", "Pneumonia"]

log(f"[INFO] X_train processed: {X_train.shape}")
log(f"[INFO] X_val processed:   {X_val.shape}")
log(f"[INFO] X_test processed:  {X_test.shape}")

class_counts = pd.DataFrame({
    "split": ["train"] * len(y_train) + ["val"] * len(y_val) + ["test"] * len(y_test),
    "label": np.concatenate([y_train, y_val, y_test])
})
dist = class_counts.groupby(["split", "label"]).size().reset_index(name="count")
dist["class_name"] = dist["label"].map({0: "Normal", 1: "Pneumonia"})
display(dist)
dist.to_csv(TABLE_DIR / "table_class_distribution.csv", index=False)
log("\n[CLASS DISTRIBUTION]")
log(dist.to_string(index=False))

In [ ]:
# ============================================================
# 6. Visual inspection of representative images
# ============================================================
plt.figure(figsize=(8, 4))
plot_idx = 1
for cls in [0, 1]:
    indices = np.where(y_train == cls)[0][:5]
    for idx in indices:
        plt.subplot(2, 5, plot_idx)
        plt.imshow(X_train[idx].squeeze(), cmap="gray")
        plt.title(CLASS_NAMES[cls])
        plt.axis("off")
        plot_idx += 1
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_representative_pneumoniamnist_images.png", dpi=300, bbox_inches="tight")
plt.show()

## Compact CNN

The model is intentionally lightweight because PneumoniaMNIST images are 28×28 grayscale images. This keeps the workflow reproducible and suitable for Colab.

In [ ]:
# ============================================================
# 7. Build compact CNN model
# ============================================================
def build_compact_cnn(input_shape=(28, 28, 1)):
    inputs = layers.Input(shape=input_shape, name="input_image")

    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="last_conv")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D(name="embedding")(x)

    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="risk_output")(x)

    model = models.Model(inputs, outputs, name="Compact_CNN_PneumoniaMNIST")
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model

cnn_model = build_compact_cnn(input_shape=X_train.shape[1:])
cnn_model.summary(print_fn=log)

In [ ]:
# ============================================================
# 8. Train CNN
# ============================================================
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
log(f"[INFO] Class weights: {class_weight}")

ckpt_path = MODEL_DIR / "best_compact_cnn.keras"

cb = [
    callbacks.ModelCheckpoint(str(ckpt_path), monitor="val_auc", mode="max", save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=8, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, verbose=1)
]

history = cnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=64,
    class_weight=class_weight,
    callbacks=cb,
    verbose=1
)

hist_df = pd.DataFrame(history.history)
hist_df.to_csv(TABLE_DIR / "table_cnn_training_history.csv", index=False)
display(hist_df.tail())

In [ ]:
# ============================================================
# 9. Plot CNN training curves
# ============================================================
plt.figure()
plt.plot(hist_df["loss"], label="Training loss")
plt.plot(hist_df["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.savefig(FIG_DIR / "figure_cnn_loss_curve.png", dpi=300, bbox_inches="tight")
plt.show()

plt.figure()
plt.plot(hist_df["accuracy"], label="Training accuracy")
plt.plot(hist_df["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Training and Validation Accuracy")
plt.legend()
plt.grid(True)
plt.savefig(FIG_DIR / "figure_cnn_accuracy_curve.png", dpi=300, bbox_inches="tight")
plt.show()

if "auc" in hist_df.columns:
    plt.figure()
    plt.plot(hist_df["auc"], label="Training AUC")
    plt.plot(hist_df["val_auc"], label="Validation AUC")
    plt.xlabel("Epoch")
    plt.ylabel("AUC")
    plt.title("CNN Training and Validation AUC")
    plt.legend()
    plt.grid(True)
    plt.savefig(FIG_DIR / "figure_cnn_auc_curve.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# 10. Evaluation utilities
# ============================================================
def expected_calibration_error_binary(y_true, prob_pos, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos).reshape(-1)
    pred = (prob_pos >= 0.5).astype(int)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    rows = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (prob_pos >= lo) & (prob_pos <= hi)
        else:
            mask = (prob_pos >= lo) & (prob_pos < hi)

        if np.any(mask):
            acc = np.mean(y_true[mask] == pred[mask])
            conf = np.mean(prob_pos[mask])
            weight = np.mean(mask)
            ece += np.abs(acc - conf) * weight
            rows.append({
                "bin_low": lo,
                "bin_high": hi,
                "count": int(np.sum(mask)),
                "accuracy": acc,
                "confidence": conf
            })

    return float(ece), pd.DataFrame(rows)

def evaluate_binary_model(name, y_true, prob_pos):
    prob_pos = np.asarray(prob_pos).reshape(-1)
    pred = (prob_pos >= 0.5).astype(int)

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, pred),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1-score": f1_score(y_true, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, prob_pos),
        "Log Loss": log_loss(y_true, np.column_stack([1 - prob_pos, prob_pos])),
        "ECE": expected_calibration_error_binary(y_true, prob_pos)[0]
    }

In [ ]:
# ============================================================
# 11. Evaluate CNN on test set
# ============================================================
cnn_prob = cnn_model.predict(X_test, verbose=0).reshape(-1)
cnn_pred = (cnn_prob >= 0.5).astype(int)

cnn_result = evaluate_binary_model("Compact CNN", y_test, cnn_prob)
cnn_results_df = pd.DataFrame([cnn_result])
display(cnn_results_df)
cnn_results_df.to_csv(TABLE_DIR / "table_cnn_test_performance.csv", index=False)

log("\n[CNN TEST PERFORMANCE]")
log(cnn_results_df.to_string(index=False))

cm = confusion_matrix(y_test, cnn_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(values_format="d")
plt.title("Confusion Matrix - Compact CNN")
plt.savefig(FIG_DIR / "figure_confusion_matrix_cnn.png", dpi=300, bbox_inches="tight")
plt.show()

report_df = pd.DataFrame(classification_report(y_test, cnn_pred, target_names=CLASS_NAMES, output_dict=True)).T
display(report_df)
report_df.to_csv(TABLE_DIR / "table_classification_report_cnn.csv")

In [ ]:
# ============================================================
# 12. ROC curve and calibration curve for CNN
# ============================================================
RocCurveDisplay.from_predictions(y_test, cnn_prob)
plt.title("ROC Curve - Compact CNN")
plt.savefig(FIG_DIR / "figure_roc_curve_cnn.png", dpi=300, bbox_inches="tight")
plt.show()

prob_true, prob_pred = calibration_curve(y_test, cnn_prob, n_bins=10)
plt.figure()
plt.plot(prob_pred, prob_true, marker="o", label="CNN")
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title("Calibration Curve - Compact CNN")
plt.legend()
plt.grid(True)
plt.savefig(FIG_DIR / "figure_calibration_curve_cnn.png", dpi=300, bbox_inches="tight")
plt.show()

ece, calib_table = expected_calibration_error_binary(y_test, cnn_prob, n_bins=10)
display(calib_table)
calib_table.to_csv(TABLE_DIR / "table_calibration_bins_cnn.csv", index=False)
log(f"[CNN] Expected Calibration Error: {ece:.5f}")

## CNN embeddings + machine-learning ensemble

The CNN is used as a feature extractor. Its embedding layer produces compact representations that are passed to classical ML models and a soft-voting ensemble.

In [ ]:
# ============================================================
# 13. Extract CNN embeddings
# ============================================================
embedding_model = models.Model(
    inputs=cnn_model.input,
    outputs=cnn_model.get_layer("embedding").output
)

Z_train = embedding_model.predict(X_train, verbose=0)
Z_val   = embedding_model.predict(X_val, verbose=0)
Z_test  = embedding_model.predict(X_test, verbose=0)

# Use train + validation for ML training after CNN selection
Z_train_full = np.vstack([Z_train, Z_val])
y_train_full = np.concatenate([y_train, y_val])

log(f"[INFO] Embedding shapes: train={Z_train.shape}, val={Z_val.shape}, test={Z_test.shape}")
np.save(OTHER_DIR / "Z_train_full_embeddings.npy", Z_train_full)
np.save(OTHER_DIR / "Z_test_embeddings.npy", Z_test)

In [ ]:
# ============================================================
# 14. Train classical ML models on CNN embeddings
# ============================================================
scale_pos_weight = (len(y_train_full) - y_train_full.sum()) / max(y_train_full.sum(), 1)
log(f"[INFO] scale_pos_weight for XGBoost: {scale_pos_weight:.3f}")

ml_models = {
    "LogisticRegression Embeddings": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED))
    ]),
    "RandomForest Embeddings": RandomForestClassifier(
        n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
    "ExtraTrees Embeddings": ExtraTreesClassifier(
        n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1
    ),
    "XGBoost Embeddings": XGBClassifier(
        n_estimators=400, learning_rate=0.03, max_depth=3,
        subsample=0.85, colsample_bytree=0.85,
        eval_metric="logloss", random_state=SEED,
        scale_pos_weight=float(scale_pos_weight)
    ),
    "LightGBM Embeddings": LGBMClassifier(
        n_estimators=400, learning_rate=0.03,
        class_weight="balanced", random_state=SEED
    )
}

ml_results = []
ml_probabilities = {}
ml_predictions = {}
trained_ml_models = {}

for name, clf in ml_models.items():
    log(f"[TRAIN] {name}")
    clf.fit(Z_train_full, y_train_full)
    prob = clf.predict_proba(Z_test)[:, 1]
    pred = (prob >= 0.5).astype(int)
    ml_results.append(evaluate_binary_model(name, y_test, prob))
    ml_probabilities[name] = prob
    ml_predictions[name] = pred
    trained_ml_models[name] = clf

ml_results_df = pd.DataFrame(ml_results).sort_values("F1-score", ascending=False)
display(ml_results_df)
ml_results_df.to_csv(TABLE_DIR / "table_embedding_ml_model_performance.csv", index=False)
log("\n[EMBEDDING ML PERFORMANCE]")
log(ml_results_df.to_string(index=False))

In [ ]:
# ============================================================
# 15. Soft-voting ensemble on embeddings
# ============================================================
soft_voting = VotingClassifier(
    estimators=[
        ("rf", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1)),
        ("et", ExtraTreesClassifier(n_estimators=400, class_weight="balanced", random_state=SEED, n_jobs=-1)),
        ("xgb", XGBClassifier(
            n_estimators=400, learning_rate=0.03, max_depth=3,
            subsample=0.85, colsample_bytree=0.85,
            eval_metric="logloss", random_state=SEED,
            scale_pos_weight=float(scale_pos_weight)
        )),
        ("lgbm", LGBMClassifier(n_estimators=400, learning_rate=0.03, class_weight="balanced", random_state=SEED))
    ],
    voting="soft"
)

log("[TRAIN] SoftVoting Embeddings")
soft_voting.fit(Z_train_full, y_train_full)

sv_prob = soft_voting.predict_proba(Z_test)[:, 1]
sv_pred = (sv_prob >= 0.5).astype(int)
sv_result = evaluate_binary_model("SoftVoting Embeddings", y_test, sv_prob)

all_results_df = pd.concat([
    cnn_results_df,
    ml_results_df,
    pd.DataFrame([sv_result])
], ignore_index=True).sort_values("F1-score", ascending=False)

display(all_results_df)
all_results_df.to_csv(TABLE_DIR / "table_all_model_performance.csv", index=False)
log("\n[ALL MODEL PERFORMANCE]")
log(all_results_df.to_string(index=False))

In [ ]:
# ============================================================
# 16. Select best model and save performance artifacts
# ============================================================
best_model_name = all_results_df.iloc[0]["Model"]
log(f"[BEST MODEL] {best_model_name}")

if best_model_name == "Compact CNN":
    best_prob = cnn_prob
    best_pred = cnn_pred
elif best_model_name == "SoftVoting Embeddings":
    best_prob = sv_prob
    best_pred = sv_pred
else:
    best_prob = ml_probabilities[best_model_name]
    best_pred = ml_predictions[best_model_name]

cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
disp.plot(values_format="d")
plt.title(f"Confusion Matrix - {best_model_name}")
plt.savefig(FIG_DIR / "figure_confusion_matrix_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

RocCurveDisplay.from_predictions(y_test, best_prob)
plt.title(f"ROC Curve - {best_model_name}")
plt.savefig(FIG_DIR / "figure_roc_curve_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

prob_true, prob_pred = calibration_curve(y_test, best_prob, n_bins=10)
plt.figure()
plt.plot(prob_pred, prob_true, marker="o", label=best_model_name)
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.title(f"Calibration Curve - {best_model_name}")
plt.legend()
plt.grid(True)
plt.savefig(FIG_DIR / "figure_calibration_curve_best_model.png", dpi=300, bbox_inches="tight")
plt.show()

best_report_df = pd.DataFrame(classification_report(y_test, best_pred, target_names=CLASS_NAMES, output_dict=True)).T
display(best_report_df)
best_report_df.to_csv(TABLE_DIR / "table_classification_report_best_model.csv")

## Case-level prediction analysis

This section saves representative correct and incorrect cases with their confidence scores.

In [ ]:
# ============================================================
# 17. Case-level correct and incorrect predictions
# ============================================================
case_df = pd.DataFrame({
    "index": np.arange(len(y_test)),
    "true_label": y_test,
    "true_class": [CLASS_NAMES[i] for i in y_test],
    "pred_label": best_pred,
    "pred_class": [CLASS_NAMES[i] for i in best_pred],
    "p_pneumonia": best_prob,
    "confidence": np.maximum(best_prob, 1 - best_prob),
    "outcome": np.where(best_pred == y_test, "Correct", "Error")
}).sort_values(["outcome", "confidence"], ascending=[True, False])

display(case_df.head(20))
case_df.to_csv(TABLE_DIR / "table_case_level_predictions.csv", index=False)

# Show top correct and top incorrect
correct_idx = case_df[case_df["outcome"] == "Correct"].head(4)["index"].tolist()
error_idx = case_df[case_df["outcome"] == "Error"].head(4)["index"].tolist()
selected = correct_idx + error_idx

plt.figure(figsize=(10, 5))
for i, idx in enumerate(selected):
    plt.subplot(2, 4, i + 1)
    plt.imshow(X_test[idx].squeeze(), cmap="gray")
    title = f"T:{CLASS_NAMES[y_test[idx]]}\nP:{CLASS_NAMES[best_pred[idx]]}\nConf:{case_df.loc[case_df['index']==idx, 'confidence'].values[0]:.2f}"
    plt.title(title, fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.savefig(FIG_DIR / "figure_case_level_correct_incorrect_predictions.png", dpi=300, bbox_inches="tight")
plt.show()

## Grad-CAM explainability

Grad-CAM is computed using the last convolutional layer of the CNN. This visualizes image regions that most strongly influence the pneumonia prediction.

In [ ]:
# ============================================================
# 18. Grad-CAM utility
# ============================================================
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="last_conv"):
    grad_model = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        class_channel = predictions[:, 0]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.reduce_max(heatmap)
    if max_val > 0:
        heatmap = heatmap / max_val
    return heatmap.numpy()

def overlay_heatmap_gray(image, heatmap, alpha=0.45):
    image = image.squeeze()
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], image.shape).numpy().squeeze()
    return image, heatmap_resized

# Select representative pneumonia and normal examples
sample_indices = []
for cls in [0, 1]:
    inds = np.where(y_test == cls)[0]
    # choose high-confidence correct if possible
    correct = [i for i in inds if best_pred[i] == y_test[i]]
    if len(correct) > 0:
        selected_idx = sorted(correct, key=lambda i: np.maximum(best_prob[i], 1-best_prob[i]), reverse=True)[0]
    else:
        selected_idx = inds[0]
    sample_indices.append(selected_idx)

plt.figure(figsize=(8, 4))
for j, idx in enumerate(sample_indices):
    img_array = X_test[idx:idx+1]
    heatmap = make_gradcam_heatmap(img_array, cnn_model, "last_conv")
    image, heatmap_resized = overlay_heatmap_gray(X_test[idx], heatmap)

    plt.subplot(2, 2, 2*j + 1)
    plt.imshow(image, cmap="gray")
    plt.title(f"Original: {CLASS_NAMES[y_test[idx]]}")
    plt.axis("off")

    plt.subplot(2, 2, 2*j + 2)
    plt.imshow(image, cmap="gray")
    plt.imshow(heatmap_resized, alpha=0.45)
    plt.title(f"Grad-CAM: P(Pneumonia)={cnn_prob[idx]:.2f}")
    plt.axis("off")

plt.tight_layout()
plt.savefig(FIG_DIR / "figure_gradcam_representative_cases.png", dpi=300, bbox_inches="tight")
plt.show()

## SHAP explanation on CNN embeddings

For the hybrid model, SHAP is applied to the Random Forest trained on CNN embeddings. These embedding-level explanations help identify which learned representation dimensions influence the pneumonia decision.

In [ ]:
# ============================================================
# 19. SHAP on CNN embeddings -- FIXED robust dimensionality
# ============================================================
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use RandomForest embeddings for stable and fast TreeExplainer analysis.
shap_model_name = "RandomForest Embeddings"
shap_model = trained_ml_models[shap_model_name]

# Limit SHAP sample size for speed.
n_shap = min(300, Z_test.shape[0])
Z_shap = Z_test[:n_shap]

feature_names = [f"emb_{i:03d}" for i in range(Z_shap.shape[1])]

def select_positive_class_shap_values(shap_values, n_samples, n_features):
    """
    Robustly converts SHAP outputs to a 2D array: (samples, features).
    Handles common SHAP formats:
    - list[class] -> each item is (samples, features)
    - ndarray (samples, features)
    - ndarray (samples, features, classes)
    - ndarray (classes, samples, features)
    - Explanation object with .values
    """
    if hasattr(shap_values, "values"):
        shap_values = shap_values.values

    if isinstance(shap_values, list):
        arr = np.asarray(shap_values[1] if len(shap_values) > 1 else shap_values[0])
    else:
        arr = np.asarray(shap_values)

    print("[SHAP] Raw shape:", arr.shape)

    if arr.ndim == 2:
        pass

    elif arr.ndim == 3:
        # Format: (samples, features, classes)
        if arr.shape[0] == n_samples and arr.shape[1] == n_features:
            cls_idx = 1 if arr.shape[2] > 1 else 0
            arr = arr[:, :, cls_idx]

        # Format: (classes, samples, features)
        elif arr.shape[1] == n_samples and arr.shape[2] == n_features:
            cls_idx = 1 if arr.shape[0] > 1 else 0
            arr = arr[cls_idx, :, :]

        # Less common fallback: squeeze if possible
        else:
            arr = np.squeeze(arr)
            if arr.ndim == 3:
                raise ValueError(f"Unsupported SHAP 3D shape after squeeze: {arr.shape}")

    else:
        arr = np.squeeze(arr)

    if arr.ndim != 2:
        raise ValueError(f"SHAP values must be 2D after correction, got shape {arr.shape}")

    if arr.shape[0] != n_samples and arr.shape[1] == n_samples:
        arr = arr.T

    if arr.shape[1] != n_features:
        raise ValueError(
            f"Feature mismatch: SHAP has {arr.shape[1]} features, "
            f"but Z_shap has {n_features} features."
        )

    print("[SHAP] Corrected shape:", arr.shape)
    return arr

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(Z_shap)

shap_values_to_plot = select_positive_class_shap_values(
    shap_values,
    n_samples=Z_shap.shape[0],
    n_features=Z_shap.shape[1]
)

log("[SHAP] SHAP values generated and corrected.")

shap.summary_plot(
    shap_values_to_plot,
    Z_shap,
    feature_names=feature_names,
    show=False
)
plt.savefig(FIG_DIR / "figure_shap_summary_embeddings.png", dpi=300, bbox_inches="tight")
plt.show()

shap.summary_plot(
    shap_values_to_plot,
    Z_shap,
    feature_names=feature_names,
    plot_type="bar",
    show=False
)
plt.savefig(FIG_DIR / "figure_shap_bar_embeddings.png", dpi=300, bbox_inches="tight")
plt.show()

# Save mean absolute SHAP importance.
mean_abs_shap = np.abs(shap_values_to_plot).mean(axis=0).reshape(-1)

if len(feature_names) != len(mean_abs_shap):
    log(
        f"[SHAP] Feature-name mismatch: {len(feature_names)} names vs "
        f"{len(mean_abs_shap)} SHAP values. Using generic names."
    )
    feature_names_fixed = [f"emb_{i:03d}" for i in range(len(mean_abs_shap))]
else:
    feature_names_fixed = feature_names

shap_importance_df = pd.DataFrame({
    "embedding_feature": feature_names_fixed,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

display(shap_importance_df.head(20))
shap_importance_df.to_csv(TABLE_DIR / "table_shap_embedding_importance.csv", index=False)

log("[SHAP] Embedding-level importance table saved.")

## Optional LIME image explanation

This cell uses LIME for image explanations. It may take a few minutes. It is useful for case-level interpretation, but Grad-CAM is usually more stable for CNN-based medical imaging.

In [ ]:
# ============================================================
# 20. Optional LIME image explanation
# ============================================================
from lime import lime_image
from skimage.segmentation import mark_boundaries

def cnn_predict_for_lime(images):
    # LIME sends RGB-like images; convert to grayscale if needed
    arr = np.array(images)
    if arr.ndim == 4 and arr.shape[-1] == 3:
        arr = arr.mean(axis=-1, keepdims=True)
    elif arr.ndim == 3:
        arr = arr[..., np.newaxis]
    arr = arr.astype("float32")
    if arr.max() > 1:
        arr = arr / 255.0
    p = cnn_model.predict(arr, verbose=0).reshape(-1)
    return np.column_stack([1 - p, p])

try:
    lime_idx = sample_indices[1] if len(sample_indices) > 1 else 0
    lime_img = X_test[lime_idx].squeeze()
    # LIME expects 3 channels for image segmentation; repeat grayscale channels
    lime_img_rgb = np.repeat(lime_img[..., np.newaxis], 3, axis=-1)

    explainer_lime = lime_image.LimeImageExplainer(random_state=SEED)
    explanation = explainer_lime.explain_instance(
        lime_img_rgb,
        cnn_predict_for_lime,
        top_labels=2,
        hide_color=0,
        num_samples=500
    )

    temp, mask = explanation.get_image_and_mask(
        label=1,
        positive_only=True,
        num_features=5,
        hide_rest=False
    )

    plt.figure(figsize=(5, 5))
    plt.imshow(mark_boundaries(temp, mask))
    plt.title("LIME Image Explanation - Pneumonia Class")
    plt.axis("off")
    plt.savefig(FIG_DIR / "figure_lime_image_explanation.png", dpi=300, bbox_inches="tight")
    plt.show()

except Exception as e:
    log(f"[WARNING] LIME image explanation failed: {e}")

In [ ]:
# ============================================================
# 21. Save models and final summary
# ============================================================
cnn_model.save(MODEL_DIR / "compact_cnn_pneumoniamnist.keras")
embedding_model.save(MODEL_DIR / "cnn_embedding_extractor.keras")
joblib.dump(trained_ml_models, MODEL_DIR / "embedding_ml_models.joblib")
joblib.dump(soft_voting, MODEL_DIR / "soft_voting_embeddings.joblib")

summary = {
    "dataset_path": DATA_PATH,
    "output_directory": str(BASE_DIR),
    "best_model": best_model_name,
    "cnn_result": cnn_result,
    "all_results": all_results_df.to_dict(orient="records")
}

with open(OTHER_DIR / "experiment_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

log("\n[FINAL SUMMARY]")
log(f"Best model: {best_model_name}")
log(f"Outputs saved to: {BASE_DIR}")
log("Notebook completed successfully.")
print(f"Done. Outputs saved to: {BASE_DIR}")

## Suggested manuscript positioning

This notebook supports a paper framing such as:

**Explainable Hybrid Deep Learning and Ensemble Decision Support for Pneumonia Detection from Benchmark Medical Images**

Recommended methodological contribution:
- A compact CNN is used to learn image representations from PneumoniaMNIST.
- Classical ensemble learning models are trained on the learned embeddings.
- Grad-CAM explains image-level attention.
- SHAP explains embedding-level decision factors.
- Calibration and case-level analysis support trust-aware interpretation.